# 06b — LSTM Model — Weekly Pure (weekly input, 1-week-ahead)

Uses **weekly-aggregated** data (W-FRI resampling, same as ARIMA/VAR/RF/XGBoost) to
predict the next week's silver log-return. Directly comparable to the other weekly models.

**Limitation**: only ~330 training sequences after aggregation and SEQ_LEN warmup.
Results should be interpreted cautiously — LSTMs typically need more data to generalise.

| Variant | Features |
|---|---|
| LSTM-Y | Weekly silver return only |
| LSTM-EXOG | Silver + weekly market returns |
| LSTM-TECH | LSTM-EXOG + macd\_line, macd\_hist, bb\_bandwidth, silver\_vol\_5w |
| LSTM-REDDIT | LSTM-EXOG + Reddit sentiment |
| LSTM-NEWS | LSTM-EXOG + news sentiment |
| LSTM-SENTIMENT | LSTM-EXOG + Reddit sentiment + news sentiment |

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings, os, sys
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 120

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: mps


## 1. Hyperparameters

In [10]:
SEQ_LEN       = 20    # lookback: 20 weeks (~5 months)
HORIZON       = 1     # 1 week ahead
HIDDEN        = 32
N_LAYERS      = 1
DROPOUT       = 0.2
EPOCHS        = 150
LR            = 1e-3
PATIENCE      = 15
BATCH         = 16    # smaller batch — fewer sequences
RETRAIN_EVERY = 4     # retrain every 4 test steps (~1 month), same cadence as RF/XGBoost

TARGET = 'silver_return'

## 2. Load & aggregate to weekly

In [11]:
train_d = pd.read_csv('../../data/processed/train.csv', index_col=0, parse_dates=True)
val_d   = pd.read_csv('../../data/processed/val.csv',   index_col=0, parse_dates=True)
test_d  = pd.read_csv('../../data/processed/test.csv',  index_col=0, parse_dates=True)

EXOG = ['gold_return', 'usd_return', 'copper_return', 'sp500_return', 'vix_return', 'oil_return']
FEAT_COLS = [TARGET] + [c for c in EXOG if c in train_d.columns]

# Aggregate: returns sum (additive), sentiment mean
def to_weekly(df):
    return df[FEAT_COLS].resample('W-FRI').sum().dropna()

train = to_weekly(train_d)
val   = to_weekly(val_d)
test  = to_weekly(test_d)

# Merge weekly sentiment if available
sent_path = '../../data/processed/daily_sentiment.csv'
if os.path.exists(sent_path):
    sent = pd.read_csv(sent_path, index_col=0, parse_dates=True)
    sent_w = sent[['reddit_sentiment','news_sentiment']].resample('W-FRI').mean()
    for df in [train, val, test]:
        for col in ['reddit_sentiment','news_sentiment']:
            df[col] = sent_w[col].reindex(df.index).ffill().fillna(0)
    print('Sentiment merged.')
else:
    for df in [train, val, test]:
        df['reddit_sentiment'] = 0.0
        df['news_sentiment']   = 0.0

print(f'Train weeks: {len(train)}  Val weeks: {len(val)}  Test weeks: {len(test)}')
print(f'Features: {train.columns.tolist()}')

Sentiment merged.
Train weeks: 365  Val weeks: 52  Test weeks: 175
Features: ['silver_return', 'gold_return', 'usd_return', 'copper_return', 'sp500_return', 'vix_return', 'oil_return', 'reddit_sentiment', 'news_sentiment']


In [12]:
# Compute the 4 weekly technical indicators (same as RF/XGBoost notebooks).
# Lagged by 1 week before joining — no lookahead.
prices   = pd.read_csv('../../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
silver_w = prices['silver'].resample('W-FRI').last().dropna()

ema_fast    = silver_w.ewm(span=12, adjust=False).mean()
ema_slow    = silver_w.ewm(span=26, adjust=False).mean()
macd_line   = ema_fast - ema_slow
macd_signal = macd_line.ewm(span=9, adjust=False).mean()

ind_w = pd.DataFrame({
    'macd_line':     macd_line,
    'macd_hist':     macd_line - macd_signal,
    'bb_bandwidth':  4 * silver_w.rolling(20).std() / silver_w.rolling(20).mean(),
    'silver_vol_5w': np.log(silver_w / silver_w.shift(1)).rolling(5).std(),
}, index=silver_w.index).shift(1)

TECH_COLS = ['macd_line', 'macd_hist', 'bb_bandwidth', 'silver_vol_5w']
for df in [train, val, test]:
    for col in TECH_COLS:
        df[col] = ind_w[col].reindex(df.index).fillna(0)

print(f'Technical indicators joined: {TECH_COLS}')

Technical indicators joined: ['macd_line', 'macd_hist', 'bb_bandwidth', 'silver_vol_5w']


## 3. Architecture & helpers

In [13]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def make_sequences(data, seq_len, target_col, horizon=1):
    X, y = [], []
    for i in range(seq_len, len(data) - horizon + 1):
        X.append(data[i - seq_len:i])
        y.append(np.sum(data[i:i + horizon, target_col]))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def run_variant(name, feature_cols):
    print(f'\n{"=" * 50}\nVariant: {name}\n{"=" * 50}')
    cols       = [c for c in feature_cols if c in train.columns]
    target_idx = cols.index(TARGET)

    scaler = StandardScaler().fit(train[cols].fillna(0))
    tr_s   = scaler.transform(train[cols].fillna(0))
    va_s   = scaler.transform(val[cols].fillna(0))
    te_s   = scaler.transform(test[cols].fillna(0))

    X_tr, y_tr = make_sequences(tr_s, SEQ_LEN, target_idx, HORIZON)
    X_va, y_va = make_sequences(va_s, SEQ_LEN, target_idx, HORIZON)
    X_te, y_te = make_sequences(te_s, SEQ_LEN, target_idx, HORIZON)
    dates      = test.index[SEQ_LEN:len(test) - HORIZON + 1]

    print(f'  Train seqs: {len(X_tr)}  Val seqs: {len(X_va)}  Test seqs: {len(X_te)}')

    def to_loader(X, y, shuffle=True):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)

    train_loader = to_loader(X_tr, y_tr)
    val_loader   = to_loader(X_va, y_va, shuffle=False)

    ckpt  = f'../../data/processed/lstm_{name.lower().replace("+","_").replace(" ","_")}_pure_weekly_best.pt'
    model = LSTMForecaster(len(cols), HIDDEN, N_LAYERS, DROPOUT).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    crit  = nn.MSELoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)

    best_val, pat_cnt = np.inf, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = np.mean([crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
                          for xb, yb in val_loader])
        sched.step(vl)
        if epoch % 25 == 0:
            print(f'  Epoch {epoch:3d}  val={vl:.6f}')
        if vl < best_val:
            best_val, pat_cnt = vl, 0
            torch.save(model.state_dict(), ckpt)
        else:
            pat_cnt += 1
            if pat_cnt >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(torch.load(ckpt))

    # Walk-forward: predict one step at a time, fine-tune every RETRAIN_EVERY steps.
    # Scaler stays fixed (fit on train only) — no lookahead.
    # Fine-tune uses expanding window of all sequences seen so far (30 epochs, LR * 0.3).
    all_scaled = np.vstack([tr_s, va_s, te_s])
    n_tr_va    = len(tr_s) + len(va_s)

    preds_list, acts_list = [], []
    for i in range(len(X_te)):
        if i > 0 and i % RETRAIN_EVERY == 0:
            X_wf, y_wf = make_sequences(all_scaled[:n_tr_va + i], SEQ_LEN, target_idx, HORIZON)
            wf_loader  = DataLoader(
                TensorDataset(torch.tensor(X_wf), torch.tensor(y_wf).unsqueeze(1)),
                batch_size=BATCH, shuffle=True)
            model.train()
            opt_r = torch.optim.Adam(model.parameters(), lr=LR * 0.3)
            for _ in range(30):
                for xb, yb in wf_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    opt_r.zero_grad()
                    crit(model(xb), yb).backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_r.step()

        model.eval()
        with torch.no_grad():
            x = torch.tensor(X_te[i:i+1]).to(DEVICE)
            preds_list.append(model(x).item())
        acts_list.append(y_te[i])

    mu, sigma  = scaler.mean_[target_idx], scaler.scale_[target_idx]
    preds      = np.array(preds_list) * sigma + HORIZON * mu
    actuals    = np.array(acts_list)  * sigma + HORIZON * mu

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    da   = np.mean(np.sign(actuals) == np.sign(preds))
    wda  = np.sum(np.abs(actuals) * (np.sign(actuals) == np.sign(preds))) / np.sum(np.abs(actuals))
    print(f'  RMSE={rmse:.6f}  MAE={mae:.6f}  DA={da:.3f}  WDA={wda:.3f}')
    print(f'  Test sequences: {len(preds)} weekly predictions')

    return {'model': name, 'rmse': rmse, 'mae': mae, 'dir_acc': da, 'wda': wda}, preds, actuals, dates

## 4. Train variants

In [14]:
MARKET_FEATURES = [TARGET] + [c for c in EXOG if c in train.columns]

variants = {
    'LSTM-Y':         [TARGET],
    'LSTM-EXOG':      MARKET_FEATURES,
    'LSTM-TECH':      MARKET_FEATURES + TECH_COLS,
    'LSTM-REDDIT':    MARKET_FEATURES + ['reddit_sentiment'],
    'LSTM-NEWS':      MARKET_FEATURES + ['news_sentiment'],
    'LSTM-SENTIMENT': MARKET_FEATURES + ['reddit_sentiment', 'news_sentiment'],
}

results     = {}
all_preds   = {}
actuals_arr = None
dates_arr   = None

for name, cols in variants.items():
    m, preds, acts, dates = run_variant(name, cols)
    results[name]   = m
    all_preds[name] = preds
    actuals_arr     = acts
    dates_arr       = dates


Variant: LSTM-Y
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 18
  RMSE=0.076898  MAE=0.055913  DA=0.497  WDA=0.476
  Test sequences: 155 weekly predictions

Variant: LSTM-EXOG
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 16
  RMSE=0.146493  MAE=0.106035  DA=0.497  WDA=0.513
  Test sequences: 155 weekly predictions

Variant: LSTM-TECH
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 19
  RMSE=0.111865  MAE=0.084798  DA=0.497  WDA=0.493
  Test sequences: 155 weekly predictions

Variant: LSTM-REDDIT
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 16
  RMSE=0.107517  MAE=0.079862  DA=0.465  WDA=0.485
  Test sequences: 155 weekly predictions

Variant: LSTM-NEWS
  Train seqs: 345  Val seqs: 32  Test seqs: 155
  Early stopping at epoch 16
  RMSE=0.096745  MAE=0.075294  DA=0.516  WDA=0.579
  Test sequences: 155 weekly predictions

Variant: LSTM-SENTIMENT
  Train seqs: 345  Val seqs: 32

## 5. Results

In [15]:
metrics_df = pd.DataFrame(list(results.values()))
metrics_df.to_csv('../../data/processed/metrics_lstm_pure_weekly.csv', index=False)

print(f'{"Model":<20}  {"RMSE":>10}  {"MAE":>10}  {"DA":>6}  {"WDA":>6}')
print('-' * 58)
for _, row in metrics_df.iterrows():
    print(f'{row["model"]:<20}  {row["rmse"]:>10.6f}  {row["mae"]:>10.6f}'
          f'  {row["dir_acc"]:>6.3f}  {row["wda"]:>6.3f}')

Model                       RMSE         MAE      DA     WDA
----------------------------------------------------------
LSTM-Y                  0.076898    0.055913   0.497   0.476
LSTM-EXOG               0.146493    0.106035   0.497   0.513
LSTM-TECH               0.111865    0.084798   0.497   0.493
LSTM-REDDIT             0.107517    0.079862   0.465   0.485
LSTM-NEWS               0.096745    0.075294   0.516   0.579
LSTM-SENTIMENT          0.113471    0.089116   0.497   0.496


## 6. Period breakdown

In [16]:
sys.path.insert(0, os.path.abspath('../../src'))
from eval_utils import period_metrics, PERIODS

best_name = max(results, key=lambda k: results[k]['wda'])
best_pred = all_preds[best_name]
print('Best variant by WDA:', best_name)

res = period_metrics(actuals_arr, best_pred, dates_arr, PERIODS)
display(res[['n', 'DA', 'WDA']].style
        .format({'n': '{:.0f}', 'DA': '{:.3f}', 'WDA': '{:.3f}'})
        .background_gradient(cmap='RdYlGn', subset=['DA', 'WDA'], vmin=0.35, vmax=0.65))
res[['n', 'DA', 'WDA']].to_csv('../../data/processed/period_lstm_pure_weekly.csv')

Best variant by WDA: LSTM-NEWS


,n,DA,WDA
Period,,,
2023 (choppy),32,0.625,0.751
2024 (bull start),52,0.500,0.544
2025 (bull run),52,0.442,0.499
2026 (YTD),19,0.579,0.589
── Full test ──,155,0.516,0.579
